<a href="https://colab.research.google.com/github/bugulin/SemEval-2026-11/blob/matus-fine-tuning/LLM_fine_tune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
HGtoken="SEM SVOJ hg WRITTING TOKEN"

In [1]:
!pip install torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 --index-url https://download.pytorch.org/whl/cu128
!pip install -q -U transformers peft accelerate trl datasets huggingface_hub
!pip install -U -q bitsandbytes>=0.46.1

Looking in indexes: https://download.pytorch.org/whl/cu128


We are downloading the needed repositaries and logging to Hugging-face, to be able to download the weihts for the Lamma model - one needs to ask for access from Metha under his token to be granted the exess and download.

In [2]:
from huggingface_hub import login


# SEM VLOŽ SVOJ HUGGING FACE TOKEN:
login(token=HGtoken)

print("Login Successful!")

Login Successful!


The model is instruction tuned one, it requires special format of the json.

"instruction": entry["syllogism"],

"response": entry["validity"]

Which is not one in sameval, this script changes the formating of the json and return well formated one for the training.

In [3]:
import json

# Define your file names
input_file = "/content/train_data.json"
output_file = "my_syllogisms.jsonl"

def convert_format(input_path, output_path):
    with open(input_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # If your JSON is a single list of dictionaries
    if not isinstance(data, list):
        data = [data]

    with open(output_path, 'w', encoding='utf-8') as f:
        for entry in data:
            # Create the new dictionary structure
            new_entry = {
                "instruction": entry["syllogism"],
                "response": str(entry["validity"]).lower()  # Converts False to "false"
            }
            # Write as JSONL (JSON Lines) which is best for training
            f.write(json.dumps(new_entry) + '\n')

    print(f"✅ Conversion complete! {len(data)} rows saved to {output_path}")

# Run the function
convert_format(input_file, output_file)

✅ Conversion complete! 960 rows saved to my_syllogisms.jsonl


We are going to download the model if it is not yet in memorry, or we are preparing it.

In [4]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

test_dataset=load_dataset("json", data_files="/content/my_syllogisms.jsonl", split="train")

# Verify we are on the GPU
print(f"Working on GPU: {torch.cuda.get_device_name(0)}")

# 4. Load Llama 3.1 8B
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token



Generating train split: 0 examples [00:00, ? examples/s]

Working on GPU: Tesla T4


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

One more formatting cell.

In [ ]:
def formatting_prompts_func(example):
    # The Trainer now passes a single example (dict) at a time
    text = (
        f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
        f"You are a logic expert specializing in syllogisms.<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n\n"
        f"For the following syllogism decide whether it is valid or not. You should respond with sigle word, without change in capitalization or puctuation, either true, if it is valid, or false otherwise. Here start your syllogism: {example['instruction']}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n"
        f"{example['response']}<|eot_id|>"
    )
    return text # Return a single string

Testing cell for the base model.

In [ ]:
import peft


num_test_silo = 50

for n in range(num_test_silo):
  test_prompt = f"For the following syllogism decide whether it is valid or not. You should respond with sigle word without change in capitalization or puctuation either true, if it id valid, or false otherwise. Here start your syllogism: {dataset[n]['instruction']}<|eot_id|>"

  messages = [
      {"role": "system", "content": "You are a logic expert specializing in syllogisms."},
      {"role": "user", "content": test_prompt},
  ]
  inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to("cuda")

  print(f"Output number: {n}\n\n")

  # Generate output
  base_output = model.generate(**inputs, max_new_tokens=50)
  print("Base Model:", tokenizer.decode(base_output[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Output number: 0




Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Base Model: system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a logic expert specializing in syllogisms.user

For the following syllogism decide whether it is valid or not. You should respond with sigle word without change in capitalization or puctuation either true, if it id valid, or false otherwise. Here start your syllogism: All cars are a type of vehicle. No animal is a car. Therefore, no animal can be a vehicle.assistant

true
Output number: 1




Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

Base Model: system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a logic expert specializing in syllogisms.user

For the following syllogism decide whether it is valid or not. You should respond with sigle word without change in capitalization or puctuation either true, if it id valid, or false otherwise. Here start your syllogism: Written works are in no way items with printed pages. Everything that is a book is an item with printed pages. Books are not written works.assistant

false
Output number: 47




Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Base Model: system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a logic expert specializing in syllogisms.user

For the following syllogism decide whether it is valid or not. You should respond with sigle word without change in capitalization or puctuation either true, if it id valid, or false otherwise. Here start your syllogism: Anything that is a plant with thorns is not a flower. A certain number of roses are considered plants with thorns. This demonstrates that some roses are not flowers.assistant

true
Output number: 48




Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Base Model: system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a logic expert specializing in syllogisms.user

For the following syllogism decide whether it is valid or not. You should respond with sigle word without change in capitalization or puctuation either true, if it id valid, or false otherwise. Here start your syllogism: Any celestial body which is a planet is a cube. A portion of celestial bodies are planets. Consequently, some celestial bodies are cubes.assistant

true
Output number: 49


Base Model: system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a logic expert specializing in syllogisms.user

For the following syllogism decide whether it is valid or not. You should respond with sigle word without change in capitalization or puctuation either true, if it id valid, or false otherwise. Here start your syllogism: Every square is a shape that is also a rectangle. Every rectangle is a polygon. It follows that some polygo

Before reruning LoRa one needs to deatach any LoRa overlay which would be added to the cell.

In [ ]:
if hasattr(model, "unload"):
    model = model.unload()


if hasattr(model, "peft_config"):
    del model.peft_config

print("✅ Model unloaded. You can now re-run your LoraConfig cell safely.")

NameError: name 'model' is not defined

This is the main training configuration cell, it is the place where all the LoRa parametres one can play with.

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

# Load your local file from the sidebar
dataset = load_dataset("json", data_files="/content/my_syllogisms.jsonl", split="train")

# 1. LoRA Configuration
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules="all-linear",
    bias="none",
    task_type="CAUSAL_LM"
)

training_args = SFTConfig(
    output_dir="./syllogism_model_checkpoints",
    max_length=512,
    dataset_text_field="text",
    packing=False,
    per_device_train_batch_size=5,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=2,
    logging_steps=1,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
)

# 3. Initialize the Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    formatting_func=formatting_prompts_func,
    args=training_args,
)


Time to train, afer everything above is set up and functioning one can trian the LoRa here. It takes some time cca 30min-hour (16GB GPU), depanding on parameters and the dataset...

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009}.


Step,Training Loss
1,3.746150
2,2.903670
3,2.149684
4,1.700025
5,1.142843
6,0.769365
7,0.666172
8,0.676000
9,0.638215
10,0.604666


TrainOutput(global_step=96, training_loss=0.47635315855344135, metrics={'train_runtime': 5130.8764, 'train_samples_per_second': 0.374, 'train_steps_per_second': 0.019, 'total_flos': 1.03207189395456e+16, 'train_loss': 0.47635315855344135})

If you run this on google colab do not forget to download your weights.

The following script tests the model´s response on the first query from the dataset.

In [ ]:
test_prompt = f"For the following syllogism decide whether it is valid or not. You should respond with sigle word, without change in capitalization or puctuation, either true, if it id valid, or false otherwise. Here start your syllogism: {dataset[0]['instruction']}<|eot_id|>" # Test with your first example

messages = [
    {"role": "system", "content": "You are a logic expert specializing in syllogisms."},
    {"role": "user", "content": test_prompt},
]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to("cuda")

# Generate output
outputs = model.generate(**inputs, max_new_tokens=100)
print("\n--- TRAINED MODEL OUTPUT ---")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.



--- TRAINED MODEL OUTPUT ---
system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a logic expert specializing in syllogisms.user

For the following syllogism decide whether it is valid or not. You should respond with sigle word, without change in capitalization or puctuation, either true, if it id valid, or false otherwise. Here start your syllogism: All cars are a type of vehicle. No animal is a car. Therefore, no animal can be a vehicle.assistant

false‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍


Formating of test dataset.

In [ ]:
import json
input_file = "/content/test_data_subtask_1.json"
output_file = "test_syllogisms.jsonl"

def convert_format(input_path, output_path):
    with open(input_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # If your JSON is a single list of dictionaries
    if not isinstance(data, list):
        data = [data]

    with open(output_path, 'w', encoding='utf-8') as f:
        for entry in data:
            # Create the new dictionary structure
            new_entry = {
                "instruction": entry["syllogism"],
                "response": str(entry["validity"]).lower()  # Converts False to "false"
            }
            # Write as JSONL (JSON Lines) which is best for training
            f.write(json.dumps(new_entry) + '\n')

    print(f"✅ Conversion complete! {len(data)} rows saved to {output_path}")

# Run the function
convert_format(input_file, output_file)

✅ Conversion complete! 191 rows saved to test_syllogisms.jsonl


First test block.

In [ ]:
import peft

test_dataset=load_dataset("json", data_files="/content/test_syllogisms.jsonl", split="train")

num_test_silo = 50
model = peft.get_peft_model(model, peft_config)

for n in range(num_test_silo):
  test_prompt = f"For the following syllogism decide whether it is valid or not. You should respond with sigle word, without change in capitalization or puctuation, either true, if it id valid, or false otherwise. Here start your syllogism: {dataset[n]['instruction']}<|eot_id|>" # Test with your first example

  messages = [
      {"role": "system", "content": "You are a logic expert specializing in syllogisms."},
      {"role": "user", "content": test_prompt},
  ]
  inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to("cuda")

  print(f"Output number{n}\n")

  # Generate output
  with model.disable_adapter():
    base_output = model.generate(**inputs, max_new_tokens=1, temperature=0.0, do_sample=False)
    print("Base Model:", tokenizer.decode(base_output[0], skip_special_tokens=True))

# 2. Enable it again (Acts like your tuned model)
# If you don't use the 'with' block, the adapter stays active by default
  tuned_output = model.generate(**inputs, max_new_tokens=1, temperature=0.0, do_sample=False)
  print("Tuned Model:", tokenizer.decode(tuned_output[0], skip_special_tokens=True))


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


Output number0



Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


Base Model: system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a logic expert specializing in syllogisms.user

For the following syllogism decide whether it is valid or not. You should respond with sigle word, without change in capitalization or puctuation, either true, if it id valid, or false otherwise. Here start your syllogism: All cars are a type of vehicle. No animal is a car. Therefore, no animal can be a vehicle.assistant

true


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


...

Base Model: system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a logic expert specializing in syllogisms.user

For the following syllogism decide whether it is valid or not. You should respond with sigle word, without change in capitalization or puctuation, either true, if it id valid, or false otherwise. Here start your syllogism: Any celestial body which is a planet is a cube. A portion of celestial bodies are planets. Consequently, some celestial bodies are cubes.assistant

false


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


Tuned Model: system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a logic expert specializing in syllogisms.user

For the following syllogism decide whether it is valid or not. You should respond with sigle word, without change in capitalization or puctuation, either true, if it id valid, or false otherwise. Here start your syllogism: Any celestial body which is a planet is a cube. A portion of celestial bodies are planets. Consequently, some celestial bodies are cubes.assistant

false
Output number49



Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


Base Model: system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a logic expert specializing in syllogisms.user

For the following syllogism decide whether it is valid or not. You should respond with sigle word, without change in capitalization or puctuation, either true, if it id valid, or false otherwise. Here start your syllogism: Every square is a shape that is also a rectangle. Every rectangle is a polygon. It follows that some polygons are squares.assistant

false
Tuned Model: system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a logic expert specializing in syllogisms.user

For the following syllogism decide whether it is valid or not. You should respond with sigle word, without change in capitalization or puctuation, either true, if it id valid, or false otherwise. Here start your syllogism: Every square is a shape that is also a rectangle. Every rectangle is a polygon. It follows that some polygons are squares.assistant

fal

In [ ]:
print(type(model))

<class 'peft.peft_model.PeftModelForCausalLM'>


In [ ]:
with model.disable_adapter():
  print(type(model))

<class 'peft.peft_model.PeftModelForCausalLM'>


The following code block anables to attach downloaded LoRa weights.

In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

adapter_path = "./LoRa_weights"
model = PeftModel.from_pretrained(model, adapter_path)


Uploading the LoRa on the Hugging face.

In [8]:
model.push_to_hub("MatusZelko/llama-3.1-syllogism-lora", token=HGtoken)
tokenizer.push_to_hub("MatusZelko/llama-3.1-syllogism-lora", token=HGtoken)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   1%|          | 1.19MB /  168MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp4opnea8w/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

CommitInfo(commit_url='https://huggingface.co/MatusZelko/llama-3.1-syllogism-lora/commit/0477326942aa0b11210c3a3c782f7acec6910790', commit_message='Upload tokenizer', commit_description='', oid='0477326942aa0b11210c3a3c782f7acec6910790', pr_url=None, repo_url=RepoUrl('https://huggingface.co/MatusZelko/llama-3.1-syllogism-lora', endpoint='https://huggingface.co', repo_type='model', repo_id='MatusZelko/llama-3.1-syllogism-lora'), pr_revision=None, pr_num=None)